In [8]:
import requests
import pandas as pd
import plotly.graph_objects as go


# 🔹 Times Série A
teams_adress_A = {
    'palmeiras': 'palmeiras/1963',
    'internacional': 'internacional/1966',
    'flamengo': 'flamengo/5981',
    'fluminense': 'fluminense/1961',
    'corinthians': 'corinthians/1957',
    'athletico paranaense': 'athletico/1967',
    'atletico mineiro': 'atletico-mineiro/1977',
    'fortaleza': 'fortaleza/2020',
    'botafogo': 'botafogo/1958',
    'santos': 'santos/1968',
    'sao paulo': 'sao-paulo/1981',
    'bragantino': 'red-bull-bragantino/1999'
}

# 🔹 Times Série B
teams_adress_B = {
    'ponte preta': 'ponte-preta/1969',
    'atletico goianiense': 'atletico-goianiense/7314',
    'america mineiro': 'america-mineiro/1973',
    'avai': 'avai/7315',
    'ceara': 'ceara/2001',
    'goias': 'goias/1960',
    'fortaleza': 'fortaleza/2020',
}

browsers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) ApplewebKit/537.36 \ (KHTML, like Gecko) Chrome / 86.0.4240.198Safari / 537.36"}


base_api = 'https://api.sofascore.com/api/v1/team/'
end_api = '/statistics/overall'


# 🔹 Detecta série
def detectar_serie(time):
    time = time.lower()
    if time in teams_adress_A:
        return 'A'
    elif time in teams_adress_B:
        return 'B'
    return None


# 🔹 API
def choose_team(time: str):

    time = time.lower()
    division = detectar_serie(time)

    if division == 'A':
        serie = '325'
        id_time = teams_adress_A[time].split('/')[-1]
        end_point_2026 = '87678'
        end_point_2025 = '72034'

    elif division == 'B':
        serie = '390'
        id_time = teams_adress_B[time].split('/')[-1]
        end_point_2026 = '89353'
        end_point_2025 = '71944'

    else:
        print(f"❌ {time} não encontrado.")
        return None

    middle_api = f'/unique-tournament/{serie}/season/'

    urls = [
        (2025, base_api + id_time + middle_api + end_point_2025 + end_api),
        (2026, base_api + id_time + middle_api + end_point_2026 + end_api)
    ]

    data_list = []

    for ano, url in urls:
        try:
            response = requests.get(url, headers=browsers, timeout=10)

            if response.status_code != 200:
                continue

            data_json = response.json()

            if "error" in data_json:
                continue

            stats = data_json.get('statistics', {})
            stats['ano'] = ano

            data_list.append(stats)

        except requests.exceptions.RequestException as e:
            print(f"❌ Erro {ano}:", e)

    return data_list


# 🔹 DataFrame
def build_dataframe(time: str):

    data = choose_team(time)

    if not data:
        return None

    df = pd.DataFrame(data)
    df = df.set_index('ano').T

    # 🔥 CORREÇÃO PRINCIPAL (anos como string)
    df.columns = df.columns.astype(str)

    # remove estruturas complexas
    df = df[~df.applymap(lambda x: isinstance(x, (dict, list))).any(axis=1)]

    # converte números
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    df['Media'] = df.mean(axis=1, numeric_only=True)

    # MultiIndex com nome do time
    df.columns = pd.MultiIndex.from_product([[time.upper()], df.columns])

    return df


# 🔹 Gera estatísticas
def gerar_estatisticas(time1: str, time2: str):

    df1 = build_dataframe(time1)
    df2 = build_dataframe(time2)

    if df1 is None or df2 is None:
        print("❌ Erro nos dados.")
        return None

    common_index = df1.index.intersection(df2.index)

    df_compare = pd.concat(
        [df1.loc[common_index], df2.loc[common_index]],
        axis=1
    )

    return df_compare


# 🔥 GRÁFICO FINAL
def gerar_grafico(df, metric: str):

    if df is None:
        print("❌ DataFrame vazio.")
        return

    # normaliza métrica
    metric_input = metric.lower()
    metric_map = {m.lower(): m for m in df.index}

    if metric_input not in metric_map:
        print(f"❌ Métrica '{metric}' não encontrada.")
        print("\n🔎 Exemplos:")
        print(list(df.index)[:10])
        return

    metric = metric_map[metric_input]

    anos = ['2025', '2026']
    times = df.columns.levels[0]

    fig = go.Figure()

    for time in times:
        valores = [
            df.loc[metric, (time, '2025')],
            df.loc[metric, (time, '2026')]
        ]

        fig.add_trace(
            go.Bar(
                name=time,
                x=anos,
                y=valores
            )
        )

    fig.update_layout(
        title=f"Comparação - {metric}",
        barmode='group',
        xaxis_title="Ano",
        yaxis_title=metric
    )

    return fig.show()

1. Gerar Dados Estatísticas  

In [9]:
df = gerar_estatisticas("sao paulo", "palmeiras")
df

C:\Users\User\AppData\Local\Temp\ipykernel_7856\1936283292.py:119: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df[~df.applymap(lambda x: isinstance(x, (dict, list))).any(axis=1)]
C:\Users\User\AppData\Local\Temp\ipykernel_7856\1936283292.py:119: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df[~df.applymap(lambda x: isinstance(x, (dict, list))).any(axis=1)]


SAO PAULO                   PALMEIRAS                  
ano                 2025     2026    Media      2025     2026    Media
goalsScored         43.0     10.0     26.5      66.0     16.0     41.0
goalsConceded       47.0      4.0     25.5      33.0      8.0     20.5
ownGoals             2.0      0.0      1.0       1.0      1.0      1.0
assists             33.0      7.0     20.0      44.0     13.0     28.5
shots              451.0     79.0    265.0     587.0     93.0    340.0
...                  ...      ...      ...       ...      ...      ...
ballRecovery      1733.0    330.0   1031.5    1832.0    286.0   1059.0
freeKicks          514.0     88.0    301.0     535.0     91.0    313.0
id               38801.0  62202.0  50501.5   38857.0  62189.0  50523.0
matches             38.0      7.0     22.5      38.0      7.0     22.5
awardedMatches       0.0      0.0      0.0       0.0      0.0      0.0

[114 rows x 6 columns]

2. Ver métricas disponíveis

In [10]:
list(df.index)[:20]

['goalsScored',
 'goalsConceded',
 'ownGoals',
 'assists',
 'shots',
 'penaltyGoals',
 'penaltiesTaken',
 'freeKickGoals',
 'freeKickShots',
 'goalsFromInsideTheBox',
 'goalsFromOutsideTheBox',
 'shotsFromInsideTheBox',
 'shotsFromOutsideTheBox',
 'headedGoals',
 'leftFootGoals',
 'rightFootGoals',
 'bigChances',
 'bigChancesCreated',
 'bigChancesMissed',
 'shotsOnTarget']

3. Gerar gráfico

In [7]:
gerar_grafico(df, "goalsScored")